# First Parallel Scripts

The Python Virtual Machine is a scalar program environment. It has only one execution thread.

If you need to execute a parallel algorithm, you need to use specific modules.
There are several modules (you can consult those modules at the following link [https://wiki.python.org/moin/ParallelProcessing]).

The main problem is that most of the Parallel Modules are OS-dependent.

One of the most compatible modules is the multiprocessing module https://docs.python.org/3.11/library/multiprocessing.html



In [ ]:
from multiprocessing import Pool
import multiprocessing as mp
import numpy as np

To analyze our code we will import those two modules:
Time module, which allows us to calculate time, like the time spent to execute our code.
cProfile, which allows us to profile and analyze our code behaviour

In [ ]:
import time
import cProfile

## NEW FOR WINDOWS PLATFORM
In the following cell we import the module where we define the functions which we will execute in the parallel threads.

*NOTE*: The following lines are commented for the first test in Windows

In [ ]:
import myfunctions as my

Every time you make changes to the Python file *myfunctions.py*, you have to reload the module to update the changes.

In [ ]:
import importlib
importlib.reload(my)

## NUMBER OF CORES

In [ ]:
MULTITHREAD=True

In [ ]:
if MULTITHREAD:
    NUMCORES=int(mp.cpu_count()/2)
else:
    NUMCORES=int(mp.cpu_count())

In [ ]:
print(f"Number of cores available: {NUMCORES}")



### First simple code: 
It implements a function that calculates the square value of a number. 
Create a sequence of numbers of length 100 million numbers, and calculate their square values.
Take the execution time.

In [ ]:
# This function is defined in the module myfunctions too.
def f(x):
    return x*x

In [ ]:
NUMDATA=1000000

In [ ]:
results_serial=np.zeros((NUMDATA,))
results_parallel=np.zeros((NUMDATA,))

In [ ]:
%%timeit -n 1 -r 10
accu =0
for index in range(0, NUMDATA):
  results_serial[index]=f(index)

Now we implement the execution in the parallel threads. Look, we invoke the f function implemented in the myfunctions module

In [ ]:
%%timeit -n 1 -r 10
with Pool(NUMCORES) as p:
    results_parallel=p.map(my.f,range(0, NUMDATA) )


In [ ]:
print(results_parallel[0:10])

Try executing this code on other machines, with more cores, and compare the results.

# Second example code:

Implement this using both sequential and parallel execution. Program a function that, given 2 numbers, calculates their product. Use this function to calculate the dot product between two random vectors with 1 million elements.



In [ ]:
NUMDATA=1000000

In [ ]:
data_X=np.random.rand(NUMDATA)
data_Y=np.random.rand(NUMDATA)

In [ ]:
# This function is now defined in the myfunctions module
# def dot(d):
#    r=d[0]*d[1]
#    return r

In [ ]:
def serial_execution(x,y):
  results_serial=np.zeros((NUMDATA,))
  data_tuple=list(zip(x,y))
  for index in range(0, len(data_tuple)):
    results_serial[index]=my.dot(data_tuple[index])
  return results_serial

In [ ]:
def parallel_execution(x,y):
  data_tuple=list(zip(x,y))
  with Pool(NUMCORES) as p:
    results_dot=p.map(my.dot,data_tuple)
  return results_dot

In [ ]:
start_time = time.time()
results_serial=serial_execution(data_X,data_Y)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
results_parallel=parallel_execution(data_X,data_Y)
print("--- %s seconds ---" % (time.time() - start_time))

Look at the times... The parallel execution isn't just bad. It is very bad!

# Profiling
We will use the profiling tools to check our code and see where the problem is!

In [ ]:
cProfile.run("serial_execution(data_X,data_Y)")

In [ ]:
cProfile.run("parallel_execution(data_X,data_Y)")

## Analysis and conclusions!

Now, we have to improve the code! 

IDEAS?



# YOUR TASK (Next Session)
Analyze how to multiply two different matrices in parallel (the use of NumPy libraries is not allowed for this specific task).

Distribute both matrices (common data) and consider two possible approaches:
* Split and compute each cell independently.
* Split by rows and compute each row in parallel (with the elements of each row calculated sequentially).

Problems to solve:
* How to submit both matrices to the library for parallel processing.
* How to submit the coordinates for the cell or row to calculate.